In [9]:
import pandas as pd
import numpy as np 
from pathlib import Path

print("Current working directory: ",Path.cwd())
current = Path.cwd()

while current.name != "pitwall" and current != current.parent:
    current = current.parent

PROJECT_ROOT = current
print("\nProject root:")
print(PROJECT_ROOT)

# Processed data directory
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "9558"

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nExists:", PROCESSED_DIR.exists())

print("Files in processed/9558")

for file in PROCESSED_DIR.iterdir():
    print(file.name)

Current working directory:  c:\Users\Nihan\.vscode\pitwall\notebooks

Project root:
c:\Users\Nihan\.vscode\pitwall

Processed directory:
c:\Users\Nihan\.vscode\pitwall\data\processed\9558

Exists: True
Files in processed/9558
car_data
laps.parquet
lap_performance.parquet
position.parquet
race_control.parquet
race_state.parquet
stints.parquet
traffic_features.parquet
weather.parquet


In [10]:
lap_performance = pd.read_parquet(
    PROCESSED_DIR / "lap_performance.parquet"
)

print("Shape:", lap_performance.shape)
print("\nColumns:")
print(lap_performance.columns.tolist())

Shape: (962, 19)

Columns:
['meeting_key', 'session_key', 'driver_number', 'lap_number', 'date_start', 'duration_sector_1', 'duration_sector_2', 'duration_sector_3', 'i1_speed', 'i2_speed', 'is_pit_out_lap', 'lap_duration', 'segments_sector_1', 'segments_sector_2', 'segments_sector_3', 'st_speed', 'stint_number', 'compound', 'tyre_age']


In [14]:
fuel_inspection = lap_performance[
    [
        "driver_number",
        "lap_number",
        "lap_duration",
        "compound",
        "tyre_age"
    ]
].copy()

print(
    fuel_inspection.head(10).to_string(index=False)
)

print("Laps per driver:")
print(
    lap_performance
    .groupby("driver_number")["lap_number"]
    .agg(["min","max","count"])
)

 driver_number  lap_number  lap_duration compound  tyre_age
             1           1        97.211   MEDIUM       0.0
             1           2        91.773   MEDIUM       1.0
             1           3        92.200   MEDIUM       2.0
             1           4        91.933   MEDIUM       3.0
             1           5        91.778   MEDIUM       4.0
             1           6        91.711   MEDIUM       5.0
             1           7        91.935   MEDIUM       6.0
             1           8        92.038   MEDIUM       7.0
             1           9        91.883   MEDIUM       8.0
             1          10        91.661   MEDIUM       9.0
Laps per driver:
               min  max  count
driver_number                 
1                1   52     52
2                1   52     52
3                1   51     51
4                1   52     52
10               1    1      1
11               1   50     50
14               1   52     52
16               1   51     51
18           

In [12]:
print("Lap duration statistics:")
print(
    lap_performance["lap_duration"].describe()
)

Lap duration statistics:
count    959.000000
mean      96.833269
std        7.612357
min       88.293000
25%       91.717000
50%       93.219000
75%      101.995000
max      128.459000
Name: lap_duration, dtype: float64


In [13]:
print("Very slow laps:")

print(
    lap_performance[
        lap_performance["lap_duration"] > 
        lap_performance["lap_duration"].quantile(0.95)
    ][
        [
            "driver_number",
            "lap_number",
            "lap_duration",
            "compound",
            "tyre_age"
        ]
    ].sort_values("lap_duration", ascending=False)
    .head(20)
    .to_string(index=False)
)

Very slow laps:
 driver_number  lap_number  lap_duration     compound  tyre_age
            23          28       128.459 INTERMEDIATE       0.0
            24          38       126.729         SOFT       0.0
            77          38       126.628         SOFT       0.0
             2          28       125.981 INTERMEDIATE       0.0
            24          20       125.955 INTERMEDIATE       0.0
            11          48       125.507         SOFT       0.0
            77          27       125.367 INTERMEDIATE       0.0
            20          28       125.299 INTERMEDIATE       0.0
            24          27       124.515 INTERMEDIATE       0.0
            14          28       124.427 INTERMEDIATE       0.0
            11          29       124.385 INTERMEDIATE       0.0
            63          28       124.342 INTERMEDIATE       0.0
            20          38       123.931         SOFT       0.0
            22          28       123.448 INTERMEDIATE       0.0
             3          

In [17]:
fuel_data = lap_performance.copy()

dry_compounds = [
    "SOFT", "MEDIUM", "HARD"
]

fuel_data = fuel_data[
    fuel_data["compound"].isin(dry_compounds)
].copy()

fuel_data = fuel_data[
    fuel_data["is_pit_out_lap"] != True
].copy()

fuel_data = fuel_data[
    fuel_data["lap_duration"] < 110
].copy()
print("Original laps:", len(lap_performance))
print("Clean fuel-estimation laps:", len(fuel_data))

print("\nCompounds:")
print(fuel_data["compound"].value_counts())

print("\nLap duration:")
print(fuel_data["lap_duration"].describe())

race_pace = (
    fuel_data
    .groupby("lap_number")["lap_duration"]
    .median()
    .reset_index()
)

print(race_pace.head(15).to_string(index=False))
print("\n...")
print(race_pace.tail(15).to_string(index=False))

fuel_data["race_phase"] = pd.cut(
    fuel_data["lap_number"],
    bins=[0,15,30,45,60],
    labels=["early", "early_mid","late_mid", "late"]
)

print(
    fuel_data
    .groupby("race_phase", observed=True)["lap_duration"]
    .agg(["count", "median", "mean"])
)


Original laps: 962
Clean fuel-estimation laps: 699

Compounds:
compound
MEDIUM    496
SOFT      162
HARD       41
Name: count, dtype: int64

Lap duration:
count    699.000000
mean      93.316632
std        3.666197
min       88.293000
25%       91.199500
50%       92.531000
75%       93.751000
max      109.747000
Name: lap_duration, dtype: float64
 lap_number  lap_duration
          1      100.7580
          2       92.9420
          3       92.6050
          4       92.9680
          5       92.6970
          6       92.6970
          7       92.7330
          8       92.7980
          9       92.5700
         10       92.6470
         11       92.4910
         12       92.4270
         13       92.6285
         14       92.4880
         15       92.5430

...
 lap_number  lap_duration
         27      107.7540
         39       92.9300
         40       91.7120
         41       90.6495
         42       90.1640
         43       90.1440
         44       90.2110
         45       90.

In [18]:
fuel_model_data = fuel_data.copy()

fuel_model_data = fuel_model_data[
    [
        "driver_number",
        "lap_number",
        "lap_duration",
        "compound",
        "tyre_age",
        "stint_number"
    ]
].copy()

fuel_model_data = fuel_model_data[
    fuel_model_data["tyre_age"] <= 25
].copy()

print("Fuel model dataset:", fuel_model_data.shape)

print("\nSample:")
print(
    fuel_model_data
    .sort_values(["driver_number", "lap_number"])
    .head(20)
    .to_string(index=False)
)

driver_compound_median = (
    fuel_model_data
    .groupby(["driver_number", "compound"])["lap_duration"]
    .transform("median")
)

fuel_model_data["normalized_lap_time"] = (
    fuel_model_data["lap_duration"] - driver_compound_median
)
print(
    fuel_model_data[
        [
            "driver_number",
            "lap_number",
            "compound",
            "tyre_age",
            "lap_duration",
            "normalized_lap_time"
        ]
    ]
    .head(20)
    .to_string(index=False)
)


Fuel model dataset: (692, 6)

Sample:
 driver_number  lap_number  lap_duration compound  tyre_age  stint_number
             1           1        97.211   MEDIUM       0.0           1.0
             1           2        91.773   MEDIUM       1.0           1.0
             1           3        92.200   MEDIUM       2.0           1.0
             1           4        91.933   MEDIUM       3.0           1.0
             1           5        91.778   MEDIUM       4.0           1.0
             1           6        91.711   MEDIUM       5.0           1.0
             1           7        91.935   MEDIUM       6.0           1.0
             1           8        92.038   MEDIUM       7.0           1.0
             1           9        91.883   MEDIUM       8.0           1.0
             1          10        91.661   MEDIUM       9.0           1.0
             1          11        91.810   MEDIUM      10.0           1.0
             1          12        91.894   MEDIUM      11.0           1.0


In [20]:
fuel_slopes = []

for (driver, compound), group in fuel_model_data.groupby(
    ["driver_number", "compound"]
):
    if len(group) < 8:
        continue

    x = group["lap_number"].values
    y = group["normalized_lap_time"].values

    slope = np.polyfit(x,y,1)[0]

    fuel_slopes.append(
        {
            "driver_number": driver,
            "compound": compound,
            "laps": len(group),
            "slope_sec_per_lap": slope
        }
    )

fuel_slopes = pd.DataFrame(fuel_slopes)
print(
    fuel_slopes
    .sort_values("slope_sec_per_lap")
    .to_string(index=False)
)



 driver_number compound  laps  slope_sec_per_lap
            31   MEDIUM    14          -0.338848
            11   MEDIUM     9          -0.287150
            77     SOFT    13          -0.224066
            23   MEDIUM    39          -0.099347
            55     HARD    10          -0.092388
            18   MEDIUM    38          -0.092324
            24     SOFT    24          -0.089457
            14   MEDIUM    39          -0.088872
            81   MEDIUM    39          -0.087855
             1     HARD    13          -0.036000
            20     SOFT    14          -0.026064
            44     SOFT    13          -0.017368
            27     SOFT    12           0.007580
            16     SOFT    13           0.028022
            22     SOFT    13           0.052945
            20   MEDIUM    26           0.078649
             4   MEDIUM    26           0.106428
             2     SOFT    13           0.107280
            22   MEDIUM    26           0.121782
             4     S

In [23]:
stint_summary = (
    fuel_model_data
    .groupby(
        ["driver_number", "stint_number", "compound"],
        dropna=False
    )
    .agg(
        laps=("lap_number", "count"),
        first_lap=("lap_number", "min"),
        last_lap=("lap_number", "max"),
        median_lap_time=("lap_duration", "median"),
        median_tyre_age=("tyre_age", "median")
    )
    .reset_index()
    .sort_values(
        ["driver_number", "stint_number"]
    )
)
print(stint_summary.to_string(index=False))

useful_stints = stint_summary[
    stint_summary["laps"] >= 8
].copy()

print(
    "Stints with at least 8 usable laps:",
    len(useful_stints)
)
print(
    useful_stints.to_string(index=False)
)

 driver_number  stint_number compound  laps  first_lap  last_lap  median_lap_time  median_tyre_age
             1           1.0   MEDIUM    26          1        26          92.0905             12.5
             1           3.0     HARD    13         40        52          89.4110              7.0
             2           1.0   MEDIUM    26          1        26          93.2290             12.5
             2           3.0     SOFT    13         40        52          91.1750              7.0
             3           1.0   MEDIUM    26          1        26          93.5475             12.5
             3           3.0     SOFT    13         39        51          91.1980              7.0
             4           1.0   MEDIUM    26          1        26          91.9360             12.5
             4           3.0     SOFT    12         41        52          89.9830              6.5
            11           1.0     HARD    18          2        19          92.7405              9.5
          

In [24]:
stint_laps = fuel_model_data.copy()

valid_stints = (
    stint_laps
    .groupby(["driver_number", "stint_number"])
    .size()
)

valid_stints = valid_stints[
    valid_stints >= 8
].index

stint_laps = stint_laps.loc[
    stint_laps.index.isin(valid_stints)
].reset_index()

print("Rows in valid stints:", len(stint_laps))

print(
    stint_laps[
        [
            "driver_number",
            "stint_number",
            "compound",
            "lap_number",
            "tyre_age",
            "lap_duration"
        ]
    ].head(30)
    .to_string(index=False)
)

Rows in valid stints: 0
Empty DataFrame
Columns: [driver_number, stint_number, compound, lap_number, tyre_age, lap_duration]
Index: []


In [26]:
tyre_slopes = []

for (driver, stint), group in stint_laps.groupby(
    ["driver_number", "stint_number"]
):
    if len(group) < 8:
        continue
    x = group["tyre_age"].values
    y = group["lap_duration"].values

    slope = np.polyfit(x, y, 1)[0]

    tyre_slopes.append(
        {
            "driver_number": driver,
            "stint_number": stint,
            "compound": group["compound"].iloc[0],
            "laps": len(group),
            "tyre_age_slope": slope
        }
    )

tyre_slopes = pd.DataFrame(tyre_slopes)

print(tyre_slopes)
print(tyre_slopes.columns.tolist())

Empty DataFrame
Columns: []
Index: []
[]


In [29]:
print("stint_laps shape:", stint_laps.shape)

print("\nColumns:")
print(stint_laps.columns.tolist())

print("\nRows per stint:")
print(
    stint_laps
    .groupby(["driver_number", "stint_number"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

print(
    stint_laps[
        [
            "driver_number",
            "stint_number",
            "compound",
            "tyre_age",
            "lap_duration"
        ]
    ].head(30).to_string(index=False)
)

print("\nTyre age dtype:")
print(stint_laps["tyre_age"].dtype)

print("\nMissing tyre age:")
print(stint_laps["tyre_age"].isna().sum())

stint_laps shape: (0, 8)

Columns:
['index', 'driver_number', 'lap_number', 'lap_duration', 'compound', 'tyre_age', 'stint_number', 'normalized_lap_time']

Rows per stint:
Series([], dtype: int64)
Empty DataFrame
Columns: [driver_number, stint_number, compound, tyre_age, lap_duration]
Index: []

Tyre age dtype:
float64

Missing tyre age:
0


In [31]:
valid_stints = (
    stint_laps
    .groupby(["driver_number", "stint_number"])
    .size()
)

valid_stints = valid_stints[
    valid_stints >= 8
].index

stint_laps = stint_laps.set_index(
    ["driver_number", "stint_number"]
)

stint_laps = stint_laps.loc[
    stint_laps.index.isin(valid_stints)
].reset_index()

# ============================================
# REBUILD VALID STINT DATA
# ============================================

# Start from the original fuel-model dataset
stint_laps = fuel_model_data.copy()

print("Starting rows:", len(stint_laps))

# Count laps in each stint
stint_counts = (
    stint_laps
    .groupby(
        ["driver_number", "stint_number"],
        dropna=False
    )
    .size()
    .reset_index(name="stint_laps_count")
)

print("\nStint counts:")
print(
    stint_counts
    .sort_values("stint_laps_count", ascending=False)
    .head(20)
    .to_string(index=False)
)

# Keep only stints with at least 8 laps
valid_stints = stint_counts[
    stint_counts["stint_laps_count"] >= 8
][
    ["driver_number", "stint_number"]
]

# Merge back instead of using MultiIndex filtering
stint_laps = stint_laps.merge(
    valid_stints,
    on=["driver_number", "stint_number"],
    how="inner"
)

print("\nRows after valid-stint filtering:", len(stint_laps))

print("\nStints remaining:", 
      stint_laps[["driver_number", "stint_number"]]
      .drop_duplicates()
      .shape[0]
)

print("\nSample:")
print(
    stint_laps[
        [
            "driver_number",
            "stint_number",
            "compound",
            "lap_number",
            "tyre_age",
            "lap_duration"
        ]
    ]
    .sort_values(["driver_number", "stint_number", "lap_number"])
    .head(30)
    .to_string(index=False)
)

Starting rows: 692

Stint counts:
 driver_number  stint_number  stint_laps_count
             1           1.0                26
             2           1.0                26
             3           1.0                26
             4           1.0                26
            23           1.0                26
            20           1.0                26
            22           1.0                26
            18           1.0                26
            14           1.0                26
            55           1.0                26
            77           1.0                26
            81           1.0                26
            27           1.0                26
            44           1.0                26
            63           1.0                26
            16           1.0                19
            31           1.0                19
            11           1.0                18
            20           3.0                14
             1           3

In [32]:
# ============================================
# WITHIN-STINT TYRE AGE TREND
# ============================================

tyre_slope_rows = []

for (driver, stint), group in stint_laps.groupby(
    ["driver_number", "stint_number"]
):

    if len(group) < 8:
        continue

    group = group.sort_values("tyre_age")

    x = group["tyre_age"].to_numpy()
    y = group["lap_duration"].to_numpy()

    slope = np.polyfit(x, y, 1)[0]

    tyre_slope_rows.append({
        "driver_number": driver,
        "stint_number": stint,
        "compound": group["compound"].iloc[0],
        "laps": len(group),
        "tyre_age_slope": slope
    })


tyre_slopes = pd.DataFrame(tyre_slope_rows)

print("Number of stints:", len(tyre_slopes))
print("Columns:", tyre_slopes.columns.tolist())

print("\nTyre-age slopes:")
print(
    tyre_slopes
    .sort_values("tyre_age_slope")
    .to_string(index=False)
)

Number of stints: 37
Columns: ['driver_number', 'stint_number', 'compound', 'laps', 'tyre_age_slope']

Tyre-age slopes:
 driver_number  stint_number compound  laps  tyre_age_slope
            11           4.0   MEDIUM     9       -0.287150
            77           3.0     SOFT    13       -0.224066
            24           1.0     SOFT    12       -0.146874
            55           3.0     HARD    10       -0.092388
            23           3.0   MEDIUM    13       -0.085940
            24           5.0     SOFT    12       -0.040336
             1           3.0     HARD    13       -0.036000
            20           3.0     SOFT    14       -0.026064
            44           3.0     SOFT    13       -0.017368
            81           3.0   MEDIUM    13       -0.015978
            18           3.0   MEDIUM    12       -0.008787
            27           3.0     SOFT    12        0.007580
            16           4.0     SOFT    13        0.028022
            14           3.0   MEDIUM   

In [33]:
field_pace = (
    stint_laps
    .groupby("lap_number")["lap_duration"]
    .agg(
        median="median",
        mean="mean",
        count="count"
    ).reset_index()
)
print(field_pace.to_string(index=False))

 lap_number   median       mean  count
          1 100.7580 100.492167     18
          2  92.9420  92.976684     19
          3  92.6050  92.719000     19
          4  92.9680  92.748053     19
          5  92.6970  92.556421     19
          6  92.6970  92.669053     19
          7  92.7330  92.889053     19
          8  92.7980  92.859263     19
          9  92.5700  92.676421     19
         10  92.6470  92.742895     19
         11  92.4910  92.593368     19
         12  92.4270  92.513158     19
         13  92.6285  92.578556     18
         14  92.4970  92.460556     18
         15  92.5660  92.468000     18
         16  93.1345  93.158500     18
         17  94.9435  95.129111     18
         18  95.5905  95.605222     18
         19 104.1205 103.845833     18
         20  97.3300  97.383667     15
         21  93.4840  93.241200     15
         22  92.4950  92.288933     15
         23  92.6120  92.635467     15
         24  93.6250  93.597600     15
         25  95.4730  95.

In [35]:
dry_model_data = stint_laps[
    (
        (stint_laps["lap_number"] <= 26) | (stint_laps["lap_number"] >= 39)
    )
].copy()

print("Dry model rows:", len(dry_model_data))

print("Race lap range:")
print(
    dry_model_data["lap_number"].agg(["min", "max"])
)

print("Race laps present:")
print(
    dry_model_data["lap_number"].unique()
)

print("Compounds")
print(
    dry_model_data["compound"].value_counts()
)

Dry model rows: 680
Race lap range:
min     1
max    52
Name: lap_number, dtype: int64
Race laps present:
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 40 41 42 43 44 45 46 47 48 49 50 51 52 39]
Compounds
compound
MEDIUM    480
SOFT      159
HARD       41
Name: count, dtype: int64


In [ ]:
import statsmodels.formula.api as smf

model = smf.ols(
    formula="""
       lap_duration
       ~ C(driver_number)
       + C(compound)
       + tyre_age
       + lap_number
    """,
    data=dry_model_data
).fit()

print(model.summary())



                            OLS Regression Results                            
Dep. Variable:           lap_duration   R-squared:                       0.375
Model:                            OLS   Adj. R-squared:                  0.354
Method:                 Least Squares   F-statistic:                     17.93
Date:                Mon, 21 Sep 2026   Prob (F-statistic):           2.18e-53
Time:                        21:58:34   Log-Likelihood:                -1617.9
No. Observations:                 680   AIC:                             3282.
Df Residuals:                     657   BIC:                             3386.
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                 91

In [38]:
print(
    model.params[
        ["tyre_age","lap_number"]
    ]
)
print("Confidence intervals:")
print(
    model.conf_int().loc[
        ["tyre_age","lap_number"]

    ]
)

print("P values:")
print(
    model.pvalues[
        ["tyre_age","lap_number"]
    ]
)

tyre_age      0.205440
lap_number   -0.084122
dtype: float64
Confidence intervals:
                   0         1
tyre_age    0.173702  0.237177
lap_number -0.099697 -0.068547
P values:
tyre_age      2.993101e-33
lap_number    2.295032e-24
dtype: float64


In [39]:
fuel_check = fuel_model_data[
    (
        (fuel_model_data["lap_number"] <= 26) | (fuel_model_data["lap_number"] >= 39)
    )
].copy()

early_compounds = set(
    fuel_check.loc[
        fuel_check["lap_number"] <= 26, "compound"
    ].dropna().unique()
)

late_compounds = set(
    fuel_check.loc[
        fuel_check["lap_number"] >= 39,
        "compound"
    ].dropna().unique()
)
common_compounds = early_compounds.intersection(late_compounds)
print("Early compounds:", early_compounds)
print("Late compounds:", late_compounds)
print("Common compounds:", common_compounds)

Early compounds: {'MEDIUM', 'SOFT', 'HARD'}
Late compounds: {'MEDIUM', 'SOFT', 'HARD'}
Common compounds: {'MEDIUM', 'SOFT', 'HARD'}


In [40]:
comparison_rows = []

for driver in fuel_check["driver_number"].unique():
    driver_data = fuel_check[
        fuel_check["driver_number"] == driver
    ]

    for compound in common_compounds:
        compound_data = driver_data[
            driver_data["compound"] == compound
        ]

        early = compound_data[
            compound_data["lap_number"] <= 26
        ]
        late = compound_data[
            compound_data["lap_number"] >= 39
        ]

        if len(early) >= 3 and len(late) >= 3:
            comparison_rows.append({
                "driver_number": driver,
                "compound": compound,
                "early_laps": len(early),
                "late_laps": len(late),
                "early_median": early["lap_duration"].median(),
                "late_median": late["lap_duration"].median(),
                "early_tyre_age_median": early["tyre_age"].median(),
                "late_tyre_age_median": late["tyre_age"].median()
            })

fuel_comparison = pd.DataFrame(comparison_rows)
fuel_comparison

,driver_number,compound,early_laps,late_laps,early_median,late_median,early_tyre_age_median,late_tyre_age_median
0,14,MEDIUM,26,13,92.7265,90.167,12.5,7.0
1,18,MEDIUM,26,12,92.8780,90.030,12.5,6.5
2,23,MEDIUM,26,13,93.0795,90.148,12.5,7.0
3,24,SOFT,12,12,94.3725,91.404,5.5,6.5
4,31,MEDIUM,3,11,98.0050,91.114,2.0,6.0
5,81,MEDIUM,26,13,91.9260,89.283,12.5,7.0


In [41]:
fuel_comparison["pace_difference"] = (
    fuel_comparison["late_median"] - fuel_comparison["early_median"]
)
fuel_comparison.sort_values(
    ["driver_number", "compound"]
)

,driver_number,compound,early_laps,late_laps,early_median,late_median,early_tyre_age_median,late_tyre_age_median,pace_difference
0,14,MEDIUM,26,13,92.7265,90.167,12.5,7.0,-2.5595
1,18,MEDIUM,26,12,92.8780,90.030,12.5,6.5,-2.8480
2,23,MEDIUM,26,13,93.0795,90.148,12.5,7.0,-2.9315
3,24,SOFT,12,12,94.3725,91.404,5.5,6.5,-2.9685
4,31,MEDIUM,3,11,98.0050,91.114,2.0,6.0,-6.8910
5,81,MEDIUM,26,13,91.9260,89.283,12.5,7.0,-2.6430


In [42]:
fuel_age_check = fuel_check.copy()

fuel_age_check["tyre_age_bucket"] = (
    fuel_age_check["tyre_age"] // 3
) * 3

fuel_age_check["race_phase"] = np.where(
    fuel_age_check["lap_number"] <= 26,
    "early",
    "late"
)

age_comparison = (
    fuel_age_check
    .groupby(
        [
            "driver_number",
            "compound",
            "tyre_age_bucket",
            "race_phase"
        ]
    )["lap_duration"]
    .median()
    .reset_index()
)

age_comparison.head(20)

,driver_number,compound,tyre_age_bucket,race_phase,lap_duration
0,1,HARD,0.0,late,89.830
1,1,HARD,3.0,late,89.372
2,1,HARD,6.0,late,89.411
3,1,HARD,9.0,late,89.286
4,1,HARD,12.0,late,89.266
5,1,MEDIUM,0.0,early,92.200
6,1,MEDIUM,3.0,early,91.778
7,1,MEDIUM,6.0,early,91.935
8,1,MEDIUM,9.0,early,91.810
9,1,MEDIUM,12.0,early,91.759


In [43]:
age_pivot = age_comparison.pivot_table(
    index=[
        "driver_number",
        "compound",
        "tyre_age_bucket"
    ],
    columns="race_phase",
    values="lap_duration"
).reset_index()

age_pivot["late_minus_early"] = (
    age_pivot["late"] - age_pivot["early"]
)
age_pivot
age_pivot["late_minus_early"].describe()

age_pivot[
    ["driver_number", "compound", "tyre_age_bucket", "late_minus_early"]
].sort_values(
    ["driver_number", "compound", "tyre_age_bucket"]
)

race_phase,driver_number,compound,tyre_age_bucket,late_minus_early
0,1,HARD,0.0,NaN
1,1,HARD,3.0,NaN
2,1,HARD,6.0,NaN
3,1,HARD,9.0,NaN
4,1,HARD,12.0,NaN
...,...,...,...,...
223,81,MEDIUM,12.0,-2.359
224,81,MEDIUM,15.0,NaN
225,81,MEDIUM,18.0,NaN
226,81,MEDIUM,21.0,NaN


In [44]:
valid_age_comparisons = age_pivot[
    age_pivot["late_minus_early"].notna()
].copy()
print("Valid comparisons:", len(valid_age_comparisons))

valid_age_comparisons[
    [
        "driver_number",
        "compound",
        "tyre_age_bucket",
        "late_minus_early"
    ]
].sort_values(
    ["driver_number", "compound", "tyre_age_bucket"]
)
valid_age_comparisons["late_minus_early"].describe()

print(
    "Median late - early difference:",
    valid_age_comparisons["late_minus_early"].median()
)

print(
    "Mean late - early difference:",
    valid_age_comparisons["late_minus_early"].mean()
)

Valid comparisons: 26
Median late - early difference: -2.6067500000000052
Mean late - early difference: -3.142134615384617


In [48]:
fuel_model_data["fuel_progression_effect"] = (
    fuel_model_data["lap_number"] * model.params["lap_number"]
)

fuel_model_data[
    [
        "driver_number",
        "lap_number",
        "compound",
        "tyre_age",
        "lap_duration",
        "fuel_progression_effect"
    ]
].head(20)

lap1_effect = model.params["lap_number"]

fuel_model_data["fuel_effect"] = (
    (fuel_model_data["lap_number"] - 1)*lap1_effect
)
fuel_model_data[
    [
        "lap_number", "fuel_effect"
    ]
].head(52)

,lap_number,fuel_effect
0,1,-0.000000
1,2,-0.084122
2,3,-0.168244
3,4,-0.252366
4,5,-0.336488
5,6,-0.420610
6,7,-0.504732
7,8,-0.588854
8,9,-0.672976
9,10,-0.757098


In [46]:
fuel_features = fuel_model_data[
    [
        "driver_number",
        "lap_number",
        "fuel_effect"
    ]
].copy()

fuel_features.head()

print("Shape:", fuel_features.shape)
print()
print(fuel_features.describe())
print()
print("Missing values:")
print(fuel_features.isna().sum())

Shape: (692, 3)

       driver_number  lap_number  fuel_effect
count     692.000000  692.000000   692.000000
mean       27.921965   23.637283    -1.904293
std        24.156921   16.583515     1.395038
min         1.000000    1.000000    -4.290221
25%        11.000000   10.000000    -3.449001
50%        22.000000   19.000000    -1.514196
75%        44.000000   42.000000    -0.757098
max        81.000000   52.000000    -0.000000

Missing values:
driver_number    0
lap_number       0
fuel_effect      0
dtype: int64


In [47]:
fuel_features_path = PROCESSED_DIR / "fuel_features.parquet"

fuel_features.to_parquet(
    fuel_features_path,
    index=False
)

print("Saved:", fuel_features_path)

Saved: c:\Users\Nihan\.vscode\pitwall\data\processed\9558\fuel_features.parquet
